<a href="https://colab.research.google.com/github/peijinUQAM/wildfire-gpd-pipeline/blob/main/gpd_ros_pipeline_v5_run_all_maps_full_biome_crosswalk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GBM-GPD Extreme Fire Spread Modeling with Partial Interpretability — Run-All Refactor

This refactored notebook is organized so a top-to-bottom **Run All** executes the full workflow in one pass: data load, preprocessing, threshold model, EBM layer, GPD-LightGBM training, baselines, spatial CV, biome CV, NGBoost comparison, SHAP, return-level outputs, and figure export.

All plots and tables are written to `output/gbm_gpd_run_all/` so the notebook can be rerun without manually copying figures.

## 1. Environment

In [21]:
!pip install -q lightgbm interpret shap google-cloud-bigquery-storage pyarrow scipy cartopy ngboost geopandas fiona pyogrio shapely

In [22]:
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import shap

from scipy.optimize import minimize_scalar
from scipy.stats import genpareto
from sklearn.metrics import mean_pinball_loss

warnings.filterwarnings("ignore")
np.random.seed(42)

print(f"LightGBM version: {lgb.__version__}")

LightGBM version: 4.6.0


## 2. Configuration

In [23]:
PROJECT_ID = "wildfireearth"
DATASET = "wildfire_analysis"
TABLE = "firegrowth_groups_v3"

RESPONSE_COL = "sprdistm"
GROUP_COL = "ecozone"
YEAR_COL = "year"
DOY_COL = "DOB"

TAU = 0.90
VALIDATION_YEARS = [2020, 2021]
EXCEEDANCE_FLOOR = 500.0
CAP_PERCENTILE = 99.0
XI_GRID = np.linspace(0.05, 0.90, 18)

ECOZONE_NAMES = {
    1: "Arctic Cordillera",
    2: "Northern Arctic",
    3: "Southern Arctic",
    4: "Taiga Plains",
    5: "Taiga Shield W",
    6: "Boreal Shield W",
    7: "Atlantic Maritime",
    8: "Mixedwood Plains",
    9: "Boreal Plains",
    10: "Prairies",
    11: "Taiga Cordillera",
    12: "Boreal Cordillera",
    13: "Pacific Maritime",
    14: "Montane Cordillera",
    15: "Hudson Plains",
    25: "Taiga Shield E",
    26: "Boreal Shield E",
}


OUTPUT_DIR = Path('output/gbm_gpd_run_all')
RUN_SPATIAL_CV = True
RUN_BIOME_CV = True
RUN_NGBOOST_COMPARISON = True
RUN_SHAP_ANALYSIS = True
RUN_RETURN_LEVELS = True
RUN_MEAN_EXCESS = True
FAST_CV = True
RANDOM_STATE = 42

RUN_CHOROPLETH_MAPS = True
ECOZONE_SHAPEFILE_PATH = None  # e.g., 'data/canada_ecozones.shp'
ECOZONE_ID_COLUMN = 'ecozone'  # column in shapefile matching notebook ecozone codes
BIOME_NAME_COLUMN = None  # optional biome name field; if None, biome map is dissolved from ecozones


## 3. Data source and extraction

In [24]:
BQ_QUERY = f"""
SELECT
  ID,
  year,
  DOB,
  ecozone,
  lat,
  lon,
  sprdistm,
  fwi, isi, ffmc, dmc, dc, bui, fwi_prev1, fwi_prev2, d_fwi, d_isi,
  ws, rh, tmax, vpd, prec, rh_prev1, rh_prev2, d_ws, d_rh, d_vpd, d_tmax, d_prec,
  Biomass, Closure, prcC, prcB, peattype, peat10,
  nonfuel1k, nonfuel2k, nonfuel5k, nonfuel10k,
  roaddens2k, roaddens10k, roaddens25k, roaddist,
  hydrodens2k, hydrodens5k, hydrodens10k, hydrodens25k,
  dem, slope, aspect_cos, aspect_sin, twi
FROM `{PROJECT_ID}.{DATASET}.{TABLE}`
WHERE sprdistm > 0
  AND sprdistm IS NOT NULL
"""

def load_from_bigquery(project_id: str = PROJECT_ID, query: str = BQ_QUERY, authenticate: bool = True) -> pd.DataFrame:
    if authenticate:
        try:
            from google.colab import auth
            auth.authenticate_user()
        except Exception:
            pass
    from google.cloud import bigquery
    client = bigquery.Client(project=project_id)
    return client.query(query).to_dataframe()

## 4. Covariates and preprocessing

In [25]:
INTERPRETED = [
    "isi",
    "fwi",
    "ws",
    "rh",
    "slope",
    "doy_sin",
    "doy_cos",
]

NON_INTERPRETED = [
    "ffmc", "dmc", "dc", "bui",
    "fwi_prev1", "fwi_prev2", "rh_prev1", "rh_prev2",
    "d_fwi", "d_isi", "d_ws", "d_rh", "d_vpd", "d_tmax", "d_prec",
    "tmax", "vpd", "prec",
    "Biomass", "Closure", "prcC", "prcB", "peattype", "peat10",
    "nonfuel1k", "nonfuel2k", "nonfuel5k", "nonfuel10k",
    "roaddens2k", "roaddens10k", "roaddens25k", "roaddist",
    "hydrodens2k", "hydrodens5k", "hydrodens10k", "hydrodens25k",
    "dem", "twi", "aspect_cos", "aspect_sin", "lat", "lon",
]

ALL_FEATURES = INTERPRETED + NON_INTERPRETED


def preprocess_fire_days(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    doy = out[DOY_COL].astype(float).fillna(1.0)
    out["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)

    if GROUP_COL in out.columns:
        out[GROUP_COL] = out[GROUP_COL].astype("category")
    if "peattype" in out.columns:
        out["peattype"] = out["peattype"].astype("category")

    usable = [c for c in ALL_FEATURES if c in out.columns]
    mask = out[usable].notna().all(axis=1) & out[RESPONSE_COL].notna()
    if GROUP_COL in out.columns:
        mask &= out[GROUP_COL].notna()

    out = out.loc[mask].reset_index(drop=True)
    return out


def summarize_dataset(df: pd.DataFrame) -> None:
    y = df[RESPONSE_COL].to_numpy(dtype=float)
    print(f"Clean dataset: {len(df):,} fire-days")
    print(f"Median spread: {np.median(y):.0f} m/day")
    print(f"95th percentile spread: {np.percentile(y, 95):.0f} m/day")
    print(f"Max spread: {np.max(y):.0f} m/day")
    print(f"Interpreted features: {len(INTERPRETED)}")
    print(f"Non-interpreted features: {len(NON_INTERPRETED)}")


## 5. Validation design

In [26]:
def build_fold_labels(ecozones: Sequence[int], min_size: int = 200) -> np.ndarray:
    ecozones = pd.Series(ecozones).astype(int)
    counts = ecozones.value_counts()
    fold_map = {int(k): int(k) for k in counts.index}

    north_zones = [1, 2, 3, 11]
    for z in north_zones:
        if z in fold_map:
            fold_map[z] = 999

    labels = np.array([fold_map.get(int(z), int(z)) for z in ecozones], dtype=int)
    return labels


def make_train_val_masks(df: pd.DataFrame, validation_years: Sequence[int] = VALIDATION_YEARS) -> Tuple[np.ndarray, np.ndarray]:
    val_mask = df[YEAR_COL].isin(validation_years).to_numpy()
    train_mask = ~val_mask
    return train_mask, val_mask


def describe_group_counts(df: pd.DataFrame) -> pd.Series:
    counts = df[GROUP_COL].astype(int).value_counts().sort_values(ascending=False)
    return counts.rename(index=ECOZONE_NAMES)


## 6. Stage 1 — adaptive threshold model

In [27]:
@dataclass
class ThresholdResult:
    model: lgb.LGBMRegressor
    threshold: np.ndarray
    exceedance_mask: np.ndarray
    credible_mask: np.ndarray
    exceedances_capped: np.ndarray
    cap_value: float


def fit_threshold_model(
    df: pd.DataFrame,
    feature_cols: Sequence[str],
    train_mask: np.ndarray,
    val_mask: np.ndarray,
    tau: float = TAU,
    exceedance_floor: float = EXCEEDANCE_FLOOR,
    cap_percentile: float = CAP_PERCENTILE,
) -> ThresholdResult:
    X = df[list(feature_cols)]
    y = df[RESPONSE_COL].to_numpy(dtype=float)

    params = dict(
        objective="quantile",
        alpha=tau,
        num_leaves=63,
        learning_rate=0.05,
        n_estimators=400,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        verbose=-1,
    )

    model = lgb.LGBMRegressor(**params)

    X_train_data = X.loc[train_mask]
    y_train_data = y[train_mask]
    X_val_data = X.loc[val_mask]
    y_val_data = y[val_mask]

    fit_args = {
        "X": X_train_data,
        "y": y_train_data,
        "init_score": np.zeros(len(y_train_data)),
        "callbacks": [lgb.early_stopping(30, verbose=False)],
    }

    if len(y_val_data) > 0:
        fit_args["eval_set"] = [(X_val_data, y_val_data)]
        fit_args["eval_init_score"] = [np.zeros(len(y_val_data))]

    model.fit(**fit_args)

    u = np.maximum(model.predict(X), 1.0)
    exceedance_mask = y > u
    credible_mask = exceedance_mask & (u >= exceedance_floor)

    y_exc = y[credible_mask] - u[credible_mask]

    if len(y_exc) == 0:
        cap_value = 0.0
        exceedances_capped = np.array([])
    else:
        cap_value = np.percentile(y_exc, cap_percentile)
        exceedances_capped = np.minimum(y_exc, cap_value)

    return ThresholdResult(
        model=model,
        threshold=u,
        exceedance_mask=exceedance_mask,
        credible_mask=credible_mask,
        exceedances_capped=exceedances_capped,
        cap_value=float(cap_value),
    )

## 7. Tail-model helpers

In [28]:
def gpd_nll_score(y_exc: np.ndarray, sigma: np.ndarray, xi: float) -> float:
    r = xi * y_exc / sigma
    denom = np.maximum(1.0 + r, 1e-10)
    nll = np.log(sigma) + (1.0 + 1.0 / xi) * np.log(denom)
    return float(np.mean(nll))


def gpd_exceedance_quantile(sigma: np.ndarray, xi: float, p: float) -> np.ndarray:
    return sigma / xi * ((1.0 - p) ** (-xi) - 1.0)


def pinball_at_level(y_exc: np.ndarray, sigma: np.ndarray, xi: float, tau: float) -> float:
    q = gpd_exceedance_quantile(sigma, xi, tau)
    return float(mean_pinball_loss(y_exc, q, alpha=tau))


def coverage_at_level(y_exc: np.ndarray, sigma: np.ndarray, xi: float, level: float) -> float:
    q = gpd_exceedance_quantile(sigma, xi, level)
    return float(np.mean(y_exc <= q))


def twcrps_score(y_exc: np.ndarray, sigma: np.ndarray, xi: float, n_quantiles: int = 50) -> float:
    taus = np.linspace(0.01, 0.999, n_quantiles)
    scores = []
    for tau in taus:
        q = gpd_exceedance_quantile(sigma, xi, tau)
        resid = y_exc - q
        pin = np.where(resid >= 0, tau * resid, (tau - 1.0) * resid)
        scores.append(np.mean(pin))
    return float(np.mean(scores))


def gpd_objective_fixed_xi(xi: float):
    def fobj(preds: np.ndarray, train_data: lgb.Dataset):
        y_true = train_data.get_label()
        sigma = np.exp(preds)
        r = xi * y_true / sigma
        denom = np.maximum(1.0 + r, 1e-8)
        c = 1.0 + 1.0 / xi
        grad = 1.0 - c * r / denom
        hess = np.maximum(c * r / denom**2, 1e-6)
        return grad, hess

    def feval(preds: np.ndarray, train_data: lgb.Dataset):
        y_true = train_data.get_label()
        sigma = np.exp(preds)
        return "gpd_nll", gpd_nll_score(y_true, sigma, xi), False

    return fobj, feval


## 8. Profile likelihood for the shape parameter

In [29]:
def profile_xi(
    X_train: np.ndarray,
    y_train_exc: np.ndarray,
    X_val: np.ndarray,
    y_val_exc: np.ndarray,
    xi_grid: np.ndarray = XI_GRID,
) -> pd.DataFrame:
    base_params = dict(
        num_leaves=63,
        learning_rate=0.05,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        verbose=-1,
    )

    sigma_init = max(float(np.mean(y_train_exc)), 1.0)
    eta_init = np.log(sigma_init)
    rows = []

    for xi in xi_grid:
        fobj, feval = gpd_objective_fixed_xi(float(xi))
        dtrain = lgb.Dataset(X_train, label=y_train_exc, init_score=np.full(len(y_train_exc), eta_init))
        dval = lgb.Dataset(X_val, label=y_val_exc, init_score=np.full(len(y_val_exc), eta_init), reference=dtrain)

        model = lgb.train(
            params={**base_params, "objective": fobj},
            train_set=dtrain,
            num_boost_round=200,
            valid_sets=[dval],
            feval=feval,
            callbacks=[lgb.early_stopping(20, verbose=False)],
        )

        rows.append({"xi": float(xi), "val_nll": float(model.best_score["valid_0"]["gpd_nll"])})

    return pd.DataFrame(rows).sort_values("val_nll").reset_index(drop=True)


## 9. EBM interpretability layer

In [30]:
from interpret.glassbox import ExplainableBoostingRegressor


def fit_ebm_layer(df: pd.DataFrame, train_mask: np.ndarray, max_rounds = 3000) -> Tuple[ExplainableBoostingRegressor, np.ndarray]:
    X_interp = df[INTERPRETED]
    y_log = np.log1p(df[RESPONSE_COL].to_numpy(dtype=float))

    ebm = ExplainableBoostingRegressor(
        feature_names=INTERPRETED,
        interactions=2,
        max_bins=256,
        max_rounds=max_rounds,
        learning_rate=0.01,
        min_samples_leaf=20,
        random_state=42,
    )
    ebm.fit(X_interp.loc[train_mask], y_log[train_mask])
    f_interp = ebm.predict(X_interp)
    return ebm, f_interp


def plot_ebm_effects(ebm: ExplainableBoostingRegressor, interpreted: Sequence[str] = INTERPRETED) -> None:
    ebm_global = ebm.explain_global()
    fig, axes = plt.subplots(1, len(interpreted), figsize=(18, 3.5))
    for i, feat in enumerate(interpreted):
        data = ebm_global.data(i)
        xvals = data["names"]
        yvals = data["scores"]
        if len(xvals) == len(yvals) + 1:
            xvals = [(xvals[j] + xvals[j + 1]) / 2 for j in range(len(yvals))]
        axes[i].plot(xvals, yvals, lw=2)
        axes[i].axhline(0, color="k", ls="--", lw=0.8)
        axes[i].set_title(feat)
    axes[0].set_ylabel("EBM effect on log(1 + spread)")
    plt.suptitle("EBM effect curves for interpreted covariates", y=1.04)
    plt.tight_layout()
    plt.show()


## 10. Tail model data assembly

In [31]:
@dataclass
class TailData:
    X_train: np.ndarray
    y_train: np.ndarray
    X_val: np.ndarray
    y_val: np.ndarray
    feature_names: List[str]
    sigma_init: float
    eta_init: float


def make_tail_data(
    df: pd.DataFrame,
    threshold_result: ThresholdResult,
    f_interp: np.ndarray,
    train_mask: np.ndarray,
    val_mask: np.ndarray,
    feature_cols: Sequence[str],
) -> TailData:
    y = df[RESPONSE_COL].to_numpy(dtype=float)
    u = threshold_result.threshold
    credible = threshold_result.credible_mask

    train_exc_mask = credible & train_mask
    val_exc_mask = credible & val_mask

    y_train_exc = np.minimum(y[train_exc_mask] - u[train_exc_mask], threshold_result.cap_value)
    y_val_exc = np.minimum(y[val_exc_mask] - u[val_exc_mask], threshold_result.cap_value)

    X_base = df[list(feature_cols)].to_numpy()
    X_train = np.hstack([X_base[train_exc_mask], f_interp[train_exc_mask].reshape(-1, 1)])
    X_val = np.hstack([X_base[val_exc_mask], f_interp[val_exc_mask].reshape(-1, 1)])

    sigma_init = max(float(np.mean(y_train_exc)), 1.0)
    eta_init = np.log(sigma_init)
    feature_names = list(feature_cols) + ["f_interp"]

    return TailData(
        X_train=X_train,
        y_train=y_train_exc,
        X_val=X_val,
        y_val=y_val_exc,
        feature_names=feature_names,
        sigma_init=sigma_init,
        eta_init=eta_init,
    )


## 11. Stage 2b — GPD-LightGBM scale model

In [32]:
@dataclass
class TailModelResult:
    xi: float
    model: lgb.Booster
    eta_init: float
    feature_names: List[str]


def fit_tail_model(tail_data: TailData, xi: float) -> TailModelResult:
    fobj, feval = gpd_objective_fixed_xi(xi)

    params = dict(
        num_leaves=127,
        learning_rate=0.03,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        verbose=-1,
        objective=fobj,
    )

    dtrain = lgb.Dataset(
        tail_data.X_train,
        label=tail_data.y_train,
        init_score=np.full(len(tail_data.y_train), tail_data.eta_init),
        feature_name=tail_data.feature_names,
    )
    dval = lgb.Dataset(
        tail_data.X_val,
        label=tail_data.y_val,
        init_score=np.full(len(tail_data.y_val), tail_data.eta_init),
        reference=dtrain,
        feature_name=tail_data.feature_names,
    )

    model = lgb.train(
        params=params,
        train_set=dtrain,
        num_boost_round=1000,
        valid_sets=[dtrain, dval],
        valid_names=["train", "val"],
        feval=feval,
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )

    return TailModelResult(
        xi=float(xi),
        model=model,
        eta_init=float(tail_data.eta_init),
        feature_names=tail_data.feature_names,
    )


def predict_sigma(model_result: TailModelResult, X_aug: np.ndarray) -> np.ndarray:
    log_sigma = model_result.eta_init + model_result.model.predict(X_aug)
    return np.exp(log_sigma)


## 12. Baselines and evaluation

In [33]:
def stationary_gpd_baseline(y_train_exc: np.ndarray) -> Tuple[float, float]:
    xi_hat, _, sigma_hat = genpareto.fit(y_train_exc, floc=0)
    return float(sigma_hat), float(xi_hat)


def fit_constant_sigma(y_train_exc: np.ndarray, xi: float) -> float:
    def objective(log_sigma: float) -> float:
        sigma = np.exp(log_sigma)
        return gpd_nll_score(y_train_exc, np.full_like(y_train_exc, sigma), xi)

    opt = minimize_scalar(objective, bounds=(np.log(1.0), np.log(10000.0)), method="bounded")
    return float(np.exp(opt.x))


def evaluate_gpd_model(y_exc: np.ndarray, sigma: np.ndarray, xi: float, label: str) -> Dict[str, float]:
    return {
        "model": label,
        "nll": gpd_nll_score(y_exc, sigma, xi),
        "twcrps": twcrps_score(y_exc, sigma, xi),
        "pinball_95": pinball_at_level(y_exc, sigma, xi, 0.95),
        "pinball_99": pinball_at_level(y_exc, sigma, xi, 0.99),
        "coverage_95": coverage_at_level(y_exc, sigma, xi, 0.95),
        "coverage_99": coverage_at_level(y_exc, sigma, xi, 0.99),
    }


def fit_quantile_baselines(
    df: pd.DataFrame,
    feature_cols: Sequence[str],
    train_mask: np.ndarray,
    val_mask: np.ndarray,
    levels: Sequence[float] = (0.95, 0.99),
) -> Dict[float, lgb.LGBMRegressor]:
    X = df[list(feature_cols)]
    y = df[RESPONSE_COL].to_numpy(dtype=float)
    models = {}
    for q in levels:
        model = lgb.LGBMRegressor(
            objective="quantile",
            alpha=q,
            num_leaves=63,
            learning_rate=0.05,
            n_estimators=400,
            min_child_samples=30,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            verbose=-1,
        )
        model.fit(
            X.loc[train_mask],
            y[train_mask],
            eval_set=[(X.loc[val_mask], y[val_mask])],
            callbacks=[lgb.early_stopping(30, verbose=False)],
        )
        models[q] = model
    return models


## 13. Return-level helper

In [34]:
def return_level(sigma: np.ndarray, xi: float, T: float, p_u: float) -> np.ndarray:
    if xi == 0:
        return sigma * np.log(T * p_u)
    return sigma / xi * ((T * p_u) ** xi - 1.0)


from pathlib import Path
from IPython.display import display

def ensure_output_dir() -> Path:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    return OUTPUT_DIR

def save_open_figure(filename: str) -> Path:
    path = ensure_output_dir() / filename
    plt.gcf().savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(plt.gcf())
    return path

def save_dataframe(df_obj: pd.DataFrame, filename: str) -> Path:
    path = ensure_output_dir() / filename
    df_obj.to_csv(path, index=False)
    return path

def evaluate_quantile_predictions(y_true: np.ndarray, y_pred: np.ndarray, tau: float) -> Dict[str, float]:
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        'model': f'Quantile LGBM {tau:.2f}',
        'pinball': float(mean_pinball_loss(y_true, y_pred, alpha=tau)),
        'coverage': float(np.mean(y_true <= y_pred)),
    }

def plot_ebm_effects_to_file(ebm: ExplainableBoostingRegressor, interpreted: Sequence[str] = INTERPRETED, filename: str = 'ebm_effects.png') -> Path:
    ebm_global = ebm.explain_global()
    fig, axes = plt.subplots(1, len(interpreted), figsize=(max(18, 3.2 * len(interpreted)), 3.5))
    if len(interpreted) == 1:
        axes = [axes]
    for i, feat in enumerate(interpreted):
        data = ebm_global.data(i)
        xvals = data['names']
        yvals = data['scores']
        if len(xvals) == len(yvals) + 1:
            xvals = [(xvals[j] + xvals[j + 1]) / 2 for j in range(len(yvals))]
        axes[i].plot(xvals, yvals, lw=2)
        axes[i].axhline(0, color='k', ls='--', lw=0.8)
        axes[i].set_title(feat)
    axes[0].set_ylabel('EBM effect on log(1 + spread)')
    fig.suptitle('EBM effect curves for interpreted covariates', y=1.04)
    fig.tight_layout()
    path = ensure_output_dir() / filename
    fig.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return path

def build_augmented_matrix(df: pd.DataFrame, f_interp: np.ndarray, feature_cols: Sequence[str] = ALL_FEATURES) -> np.ndarray:
    return np.hstack([df[list(feature_cols)].to_numpy(), np.asarray(f_interp).reshape(-1, 1)])

def compute_main_results() -> Dict[str, object]:
    ensure_output_dir()
    df_raw = load_from_bigquery()
    df = preprocess_fire_days(df_raw)
    summarize_dataset(df)

    train_mask, val_mask = make_train_val_masks(df)
    threshold_result = fit_threshold_model(df, ALL_FEATURES, train_mask, val_mask)
    ebm, f_interp = fit_ebm_layer(df, train_mask)
    plot_ebm_effects_to_file(ebm)

    tail_data = make_tail_data(df, threshold_result, f_interp, train_mask, val_mask, ALL_FEATURES)

    # Check if there's enough data for GPD tail modeling
    min_gpd_samples = 50
    can_fit_gpd_models = len(tail_data.y_train) >= min_gpd_samples and len(tail_data.y_val) >= min_gpd_samples

    xi_profile = pd.DataFrame()
    xi_best = np.nan
    tail_model = None
    sigma_all = np.full(len(df), np.nan)
    stationary_sigma, stationary_xi, const_sigma = np.nan, np.nan, np.nan
    evaluation_df = pd.DataFrame()

    if can_fit_gpd_models:
        xi_profile = profile_xi(
            tail_data.X_train,
            tail_data.y_train,
            tail_data.X_val,
            tail_data.y_val,
        )

        if not xi_profile.empty:
            xi_best = float(xi_profile.iloc[0]['xi'])
            save_dataframe(xi_profile, 'xi_profile.csv')

            tail_model = fit_tail_model(tail_data, xi_best)
            X_all_aug = build_augmented_matrix(df, f_interp, ALL_FEATURES)
            sigma_all = predict_sigma(tail_model, X_all_aug)

            y = df[RESPONSE_COL].to_numpy(dtype=float)
            credible_mask = threshold_result.credible_mask
            train_exc_mask = credible_mask & train_mask
            val_exc_mask = credible_mask & val_mask
            y_train_exc = np.minimum(y[train_exc_mask] - threshold_result.threshold[train_exc_mask], threshold_result.cap_value)
            y_val_exc = np.minimum(y[val_exc_mask] - threshold_result.threshold[val_exc_mask], threshold_result.cap_value)
            sigma_val = sigma_all[val_exc_mask]

            stationary_sigma, stationary_xi = stationary_gpd_baseline(y_train_exc)
            const_sigma = fit_constant_sigma(y_train_exc, xi_best)
            full_eval = evaluate_gpd_model(y_val_exc, sigma_val, xi_best, label=f'GBM-GPD fixed xi={xi_best:.2f}')
            b1_eval = evaluate_gpd_model(y_val_exc, np.full_like(y_val_exc, stationary_sigma), stationary_xi, label='Stationary GPD')
            b3_eval = evaluate_gpd_model(y_val_exc, np.full_like(y_val_exc, const_sigma), xi_best, label='Const-sigma GPD')
            evaluation_df = pd.DataFrame([b1_eval, b3_eval, full_eval])
            save_dataframe(evaluation_df, 'gpd_baseline_comparison.csv')
        else:
            print("Warning: xi_profile was empty, skipping GPD model fitting and evaluation.")
    else:
        print(f"Warning: Not enough samples for GPD tail modeling (train: {len(tail_data.y_train)}, val: {len(tail_data.y_val)}). Skipping GPD model fitting and evaluation.")

    quantile_models = fit_quantile_baselines(df, ALL_FEATURES, train_mask, val_mask, levels=(0.95, 0.99))
    y_val = df[RESPONSE_COL].to_numpy(dtype=float)[val_mask]
    q_rows = []
    for tau, model in quantile_models.items():
        q_rows.append(evaluate_quantile_predictions(y_val, model.predict(df.loc[val_mask, ALL_FEATURES]), tau))
    quantile_df = pd.DataFrame(q_rows)
    save_dataframe(quantile_df, 'quantile_baseline_comparison.csv')

    state = {
        'df_raw': df_raw,
        'df': df,
        'train_mask': train_mask,
        'val_mask': val_mask,
        'threshold_result': threshold_result,
        'ebm': ebm,
        'f_interp': f_interp,
        'tail_data': tail_data,
        'xi_profile': xi_profile,
        'xi_best': xi_best,
        'tail_model': tail_model,
        'X_all_aug': build_augmented_matrix(df, f_interp, ALL_FEATURES) if can_fit_gpd_models else np.array([]),
        'sigma_all': sigma_all,
        'y': df[RESPONSE_COL].to_numpy(dtype=float),
        'credible_mask': threshold_result.credible_mask,
        'train_exc_mask': threshold_result.credible_mask & train_mask,
        'val_exc_mask': threshold_result.credible_mask & val_mask,
        'y_train_exc': tail_data.y_train,
        'y_val_exc': tail_data.y_val,
        'evaluation_df': evaluation_df,
        'quantile_df': quantile_df,
        'stationary_sigma': stationary_sigma,
        'stationary_xi': stationary_xi,
        'const_sigma': const_sigma,
        'can_fit_gpd_models': can_fit_gpd_models
    }
    return state

In [35]:
from pathlib import Path
from IPython.display import display

def ensure_output_dir() -> Path:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    return OUTPUT_DIR

def save_open_figure(filename: str) -> Path:
    path = ensure_output_dir() / filename
    plt.gcf().savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(plt.gcf())
    return path

def save_dataframe(df_obj: pd.DataFrame, filename: str) -> Path:
    path = ensure_output_dir() / filename
    df_obj.to_csv(path, index=False)
    return path

def evaluate_quantile_predictions(y_true: np.ndarray, y_pred: np.ndarray, tau: float) -> Dict[str, float]:
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        'model': f'Quantile LGBM {tau:.2f}',
        'pinball': float(mean_pinball_loss(y_true, y_pred, alpha=tau)),
        'coverage': float(np.mean(y_true <= y_pred)),
    }

def plot_ebm_effects_to_file(ebm: ExplainableBoostingRegressor, interpreted: Sequence[str] = INTERPRETED, filename: str = 'ebm_effects.png') -> Path:
    ebm_global = ebm.explain_global()
    fig, axes = plt.subplots(1, len(interpreted), figsize=(max(18, 3.2 * len(interpreted)), 3.5))
    if len(interpreted) == 1:
        axes = [axes]
    for i, feat in enumerate(interpreted):
        data = ebm_global.data(i)
        xvals = data['names']
        yvals = data['scores']
        if len(xvals) == len(yvals) + 1:
            xvals = [(xvals[j] + xvals[j + 1]) / 2 for j in range(len(yvals))]
        axes[i].plot(xvals, yvals, lw=2)
        axes[i].axhline(0, color='k', ls='--', lw=0.8)
        axes[i].set_title(feat)
    axes[0].set_ylabel('EBM effect on log(1 + spread)')
    fig.suptitle('EBM effect curves for interpreted covariates', y=1.04)
    fig.tight_layout()
    path = ensure_output_dir() / filename
    fig.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return path

def build_augmented_matrix(df: pd.DataFrame, f_interp: np.ndarray, feature_cols: Sequence[str] = ALL_FEATURES) -> np.ndarray:
    return np.hstack([df[list(feature_cols)].to_numpy(), np.asarray(f_interp).reshape(-1, 1)])

def compute_main_results() -> Dict[str, object]:
    ensure_output_dir()
    df_raw = load_from_bigquery()
    df = preprocess_fire_days(df_raw)
    summarize_dataset(df)

    train_mask, val_mask = make_train_val_masks(df)
    threshold_result = fit_threshold_model(df, ALL_FEATURES, train_mask, val_mask)
    ebm, f_interp = fit_ebm_layer(df, train_mask)
    plot_ebm_effects_to_file(ebm)

    tail_data = make_tail_data(df, threshold_result, f_interp, train_mask, val_mask, ALL_FEATURES)
    xi_profile = profile_xi(
    tail_data.X_train,
    tail_data.y_train,
    tail_data.X_val,
    tail_data.y_val,)

    xi_best = float(xi_profile.iloc[0]['xi'])
    save_dataframe(xi_profile, 'xi_profile.csv')

    tail_model = fit_tail_model(tail_data, xi_best)
    X_all_aug = build_augmented_matrix(df, f_interp, ALL_FEATURES)
    sigma_all = predict_sigma(tail_model, X_all_aug)

    y = df[RESPONSE_COL].to_numpy(dtype=float)
    credible_mask = threshold_result.credible_mask
    train_exc_mask = credible_mask & train_mask
    val_exc_mask = credible_mask & val_mask
    y_train_exc = np.minimum(y[train_exc_mask] - threshold_result.threshold[train_exc_mask], threshold_result.cap_value)
    y_val_exc = np.minimum(y[val_exc_mask] - threshold_result.threshold[val_exc_mask], threshold_result.cap_value)
    sigma_val = sigma_all[val_exc_mask]

    stationary_sigma, stationary_xi = stationary_gpd_baseline(y_train_exc)
    const_sigma = fit_constant_sigma(y_train_exc, xi_best)
    full_eval = evaluate_gpd_model(y_val_exc, sigma_val, xi_best, label=f'GBM-GPD fixed xi={xi_best:.2f}')
    b1_eval = evaluate_gpd_model(y_val_exc, np.full_like(y_val_exc, stationary_sigma), stationary_xi, label='Stationary GPD')
    b3_eval = evaluate_gpd_model(y_val_exc, np.full_like(y_val_exc, const_sigma), xi_best, label='Const-sigma GPD')

    quantile_models = fit_quantile_baselines(df, ALL_FEATURES, train_mask, val_mask, levels=(0.95, 0.99))
    y_val = y[val_mask]
    q_rows = []
    for tau, model in quantile_models.items():
        q_rows.append(evaluate_quantile_predictions(y_val, model.predict(df.loc[val_mask, ALL_FEATURES]), tau))
    evaluation_df = pd.DataFrame([b1_eval, b3_eval, full_eval])
    quantile_df = pd.DataFrame(q_rows)
    save_dataframe(evaluation_df, 'gpd_baseline_comparison.csv')
    save_dataframe(quantile_df, 'quantile_baseline_comparison.csv')

    state = {
        'df_raw': df_raw,
        'df': df,
        'train_mask': train_mask,
        'val_mask': val_mask,
        'threshold_result': threshold_result,
        'ebm': ebm,
        'f_interp': f_interp,
        'tail_data': tail_data,
        'xi_profile': xi_profile,
        'xi_best': xi_best,
        'tail_model': tail_model,
        'X_all_aug': X_all_aug,
        'sigma_all': sigma_all,
        'y': y,
        'credible_mask': credible_mask,
        'train_exc_mask': train_exc_mask,
        'val_exc_mask': val_exc_mask,
        'y_train_exc': y_train_exc,
        'y_val_exc': y_val_exc,
        'evaluation_df': evaluation_df,
        'quantile_df': quantile_df,
        'stationary_sigma': stationary_sigma,
        'stationary_xi': stationary_xi,
        'const_sigma': const_sigma,
    }
    return state


## 15. Secondary analyses

In [36]:
BIOME_MAP = {
    1: 'Arctic',
    2: 'Arctic',
    3: 'Arctic',
    4: 'Taiga',
    5: 'Taiga',
    6: 'Boreal',
    7: 'Temperate/Maritime',
    8: 'Temperate/Mixedwood',
    9: 'Boreal',
    10: 'Prairie/Grassland',
    11: 'Taiga',
    12: 'Boreal',
    13: 'Temperate/Maritime',
    14: 'Cordillera',
    15: 'Hudson Plains',
    25: 'Taiga',
    26: 'Boreal',
    999: 'Taiga',
}


def run_mean_excess_analysis(state: Dict[str, object]) -> pd.DataFrame:
    y_sorted = np.sort(np.asarray(state['y_train_exc'], dtype=float))
    rows = []
    for q in np.linspace(0.10, 0.95, 40):
        threshold = np.quantile(y_sorted, q)
        exc = y_sorted[y_sorted > threshold] - threshold
        if len(exc) >= 30:
            rows.append({'threshold': threshold, 'empirical_mean_excess': exc.mean()})
    mean_excess_df = pd.DataFrame(rows)
    mean_excess_df['gpd_mean_excess'] = state['stationary_sigma'] + state['stationary_xi'] * mean_excess_df['threshold']
    mean_excess_df['gpd_mean_excess'] = mean_excess_df['gpd_mean_excess'] / max(1e-6, (1 - state['stationary_xi']))
    plt.figure(figsize=(7, 4))
    plt.plot(mean_excess_df['threshold'], mean_excess_df['empirical_mean_excess'], 'o-', ms=4, label='Empirical')
    plt.plot(mean_excess_df['threshold'], mean_excess_df['gpd_mean_excess'], 'r--', label='Stationary GPD')
    plt.xlabel('Threshold exceedance (m/day)')
    plt.ylabel('Mean excess')
    plt.title('Mean excess plot')
    plt.legend()
    save_open_figure('mean_excess_plot.png')
    save_dataframe(mean_excess_df, 'mean_excess_plot_data.csv')
    return mean_excess_df

def _fit_single_fold(df: pd.DataFrame, train_mask: np.ndarray, test_mask: np.ndarray, xi_fixed: float) -> Dict[str, float] | None:
    threshold_result = fit_threshold_model(df, ALL_FEATURES, train_mask, test_mask)
    ebm, f_interp = fit_ebm_layer(df, train_mask, max_rounds = 500 if FAST_CV else 3000)
    tail_data = make_tail_data(df, threshold_result, f_interp, train_mask, test_mask, ALL_FEATURES)
    if len(tail_data.y_train) < 50 or len(tail_data.y_val) < 50:
      return None

    model = fit_tail_model(tail_data, xi_fixed)
    y_test_exc = tail_data.y_val
    sigma_test = predict_sigma(model, tail_data.X_val)
    out = evaluate_gpd_model(y_test_exc, sigma_test, xi_fixed, label='GBM-GPD')
    out['n_test'] = int(len(y_test_exc))
    return out

def run_spatial_block_cv(state: Dict[str, object]) -> pd.DataFrame:
    df = state['df']
    labels = build_fold_labels(df[GROUP_COL].astype(int))
    rows = []
    for fold in sorted(pd.unique(labels)):
        test_mask = labels == fold
        train_mask = ~test_mask
        metrics = _fit_single_fold(df, train_mask, test_mask, state['xi_best'])
        if metrics is None:
            continue
        metrics['fold'] = int(fold)
        metrics['group'] = ECOZONE_NAMES.get(int(fold), f'Zone {int(fold)}')
        rows.append(metrics)
    cv_df = pd.DataFrame(rows)
    save_dataframe(cv_df, 'spatial_block_cv.csv')
    if not cv_df.empty:
        plot_df = cv_df.sort_values('twcrps')
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(plot_df['group'], plot_df['twcrps'])
        plt.xlabel('twCRPS')
        plt.ylabel('Held-out ecozone')
        plt.title('Spatial block CV (ecozone holdout)')
        save_open_figure('spatial_block_cv_twcrps.png')
    return cv_df

def run_biome_cv(state: Dict[str, object]) -> pd.DataFrame:
    df = state['df'].copy()
    df['biome'] = df[GROUP_COL].astype(int).map(BIOME_MAP).fillna('Other')
    rows = []
    for biome in sorted(df['biome'].unique()):
        test_mask = (df['biome'] == biome).to_numpy()
        train_mask = ~test_mask
        metrics = _fit_single_fold(df, train_mask, test_mask, state['xi_best'])
        if metrics is None:
            continue
        metrics['biome'] = biome
        rows.append(metrics)
    biome_df = pd.DataFrame(rows)
    save_dataframe(biome_df, 'biome_cv.csv')
    if not biome_df.empty:
        plot_df = biome_df.sort_values('twcrps')
        plt.figure(figsize=(8, max(4, 0.5 * len(plot_df))))
        plt.barh(plot_df['biome'], plot_df['twcrps'], color='tab:orange')
        plt.xlabel('twCRPS')
        plt.ylabel('Held-out biome')
        plt.title('Biome CV holdout performance')
        save_open_figure('biome_cv_twcrps.png')
    return biome_df

def compare_cv_schemes(spatial_df: pd.DataFrame, biome_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for name, df_ in [('Ecozone-based CV', spatial_df), ('Biome-based CV', biome_df)]:
        if df_.empty:
            continue
        rows.append({
            'Scheme': name,
            'Mean twCRPS': float(df_['twcrps'].mean()),
            'Std twCRPS': float(df_['twcrps'].std()),
            'Min twCRPS': float(df_['twcrps'].min()),
            'Max twCRPS': float(df_['twcrps'].max()),
            'Total Test Samples': int(df_['n_test'].sum()),
        })
    out = pd.DataFrame(rows)
    save_dataframe(out, 'cv_scheme_comparison.csv')
    if not out.empty:
        plt.figure(figsize=(7, 5))
        plt.bar(out['Scheme'], out['Mean twCRPS'], yerr=out['Std twCRPS'], capsize=8)
        plt.ylabel('Mean twCRPS')
        plt.title('Ecozone vs biome CV')
        save_open_figure('cv_scheme_comparison.png')
    return out

def run_ngboost_comparison(state: Dict[str, object]) -> pd.DataFrame:
    from ngboost import NGBRegressor
    from ngboost.distns.distn import RegressionDistn
    from ngboost.scores import LogScore
    from sklearn.tree import DecisionTreeRegressor

    class GPDLogScore(LogScore):
        def score(self, Y):
            return -self.dist.logpdf(Y)

        def d_score(self, Y):
            D = np.zeros((len(Y), 2))
            sigma = self.dist.scale
            xi = self.dist.xi
            r = xi * Y / sigma
            denom = np.maximum(1.0 + r, 1e-8)
            c = 1.0 + 1.0 / xi
            D[:, 0] = 1.0 - c * r / denom
            d_nll_d_xi = -1.0 / (xi ** 2) * np.log(denom) + c * Y / sigma / denom
            sig_raw = 1.0 / (1.0 + np.exp(-self.dist.raw_xi))
            d_xi_d_raw = 1.49 * sig_raw * (1.0 - sig_raw)
            D[:, 1] = d_nll_d_xi * d_xi_d_raw
            return D
    class GPDDist(RegressionDistn):
      n_params = 2
      scores = [GPDLogScore]

      def __init__(self, params):
          self._params_raw = params
          self.log_sigma = params[0]
          self.raw_xi = params[1]
          self.scale = np.maximum(np.exp(self.log_sigma), 1e-6)
          self.xi = np.clip(0.01 + 1.49 / (1.0 + np.exp(-self.raw_xi)), 0.01, 1.49)
          self._scipy_dist = genpareto(c=self.xi, loc=0, scale=self.scale)  # ← renamed

      @staticmethod
      def fit(Y):
          xi_fit, _, sigma_fit = genpareto.fit(Y, floc=0)
          xi_fit = float(np.clip(xi_fit, 0.02, 1.48))
          sigma_fit = float(max(sigma_fit, 1.0))
          log_sigma = np.log(sigma_fit)
          raw_xi = np.log((xi_fit - 0.01) / (1.50 - xi_fit))
          return np.array([log_sigma, raw_xi])

      def sample(self, m):
          return np.array([self._scipy_dist.rvs() for _ in range(m)])  # ← updated

      @property
      def params(self):
          return {'scale': self.scale, 'xi': self.xi}

    X_train = state['X_all_aug'][state['train_exc_mask']]
    X_val = state['X_all_aug'][state['val_exc_mask']]
    y_train = np.maximum(np.asarray(state['y_train_exc'], dtype=float), 0.1)
    y_val = np.maximum(np.asarray(state['y_val_exc'], dtype=float), 0.1)

    ngb = NGBRegressor(
        Dist=GPDDist,
        Score=GPDLogScore,
        Base=DecisionTreeRegressor(max_depth=4, min_samples_leaf=20),
        n_estimators=1000,
        learning_rate=0.03,
        natural_gradient=True,
        minibatch_frac=0.8,
        random_state=RANDOM_STATE,
        verbose=False,
    )
    ngb.fit(X_train, y_train, X_val=X_val, Y_val=y_val, early_stopping_rounds=50)

    dist = ngb.pred_dist(X_val)
    sigma_ngb = np.asarray(dist.scale, dtype=float)
    xi_ngb = np.asarray(dist.xi, dtype=float)

    out = pd.DataFrame([{
        'model': 'NGBoost-GPD learned xi',
        'nll': float((-dist.logpdf(y_val)).mean()),
        'twcrps': float(twcrps_score(y_val, sigma_ngb, float(np.median(xi_ngb)))),
        'pinball95': float(pinball_at_level(y_val, sigma_ngb, float(np.median(xi_ngb)), 0.95)),
        'pinball99': float(pinball_at_level(y_val, sigma_ngb, float(np.median(xi_ngb)), 0.99)),
        'coverage95': float(coverage_at_level(y_val, sigma_ngb, float(np.median(xi_ngb)), 0.95)),
        'coverage99': float(coverage_at_level(y_val, sigma_ngb, float(np.median(xi_ngb)), 0.99)),
        'median_xi': float(np.median(xi_ngb)),
        'mean_xi': float(np.mean(xi_ngb)),
    }])
    save_dataframe(out, 'ngboost_comparison.csv')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(xi_ngb, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[0].axvline(state['xi_best'], color='red', ls='--', lw=2, label=f'GBM-GPD fixed xi={state["xi_best"]:.2f}')
    axes[0].axvline(np.median(xi_ngb), color='orange', lw=2, label=f'NGBoost median={np.median(xi_ngb):.2f}')
    axes[0].set_xlabel('xi')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Distribution of NGBoost xi')
    axes[0].legend(fontsize=9)

    axes[1].scatter(sigma_ngb, xi_ngb, s=8, alpha=0.4, color='steelblue')
    axes[1].axhline(state['xi_best'], color='red', ls='--', lw=1.5)
    axes[1].set_xlabel('sigma')
    axes[1].set_ylabel('xi')
    axes[1].set_title('NGBoost sigma vs xi')

    fig.tight_layout()
    fig.savefig(ensure_output_dir() / 'ngboost_xi_distribution.png', dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return out

def run_shap_analysis(state: Dict[str, object]) -> pd.DataFrame:
    X = state['X_all_aug'][state['credible_mask']]
    feature_names = list(ALL_FEATURES) + ['f_interp']
    if len(X) > 3000:
        rng = np.random.default_rng(RANDOM_STATE)
        X = X[rng.choice(len(X), size=3000, replace=False)]
    explainer = shap.TreeExplainer(state['tail_model'].model)
    shap_values = explainer.shap_values(X)
    mean_abs = np.abs(shap_values).mean(axis=0)
    shap_df = pd.DataFrame({'feature': feature_names, 'mean_abs_shap': mean_abs}).sort_values('mean_abs_shap', ascending=False)
    save_dataframe(shap_df, 'shap_importance.csv')
    shap.summary_plot(shap_values, X, feature_names=feature_names, show=False)
    save_open_figure('shap_summary_beeswarm.png')
    shap.summary_plot(shap_values, X, feature_names=feature_names, plot_type='bar', show=False)
    save_open_figure('shap_summary_bar.png')
    return shap_df

def run_return_level_analysis(state: Dict[str, object]) -> pd.DataFrame:
    df = state['df'].copy()
    sigma_all = np.asarray(state['sigma_all'], dtype=float)
    p_u = 1.0 - TAU
    out = pd.DataFrame({'ecozone': df[GROUP_COL].astype(int)})
    for T in (2, 5, 10, 20):
        out[f'rl_{T}yr'] = return_level(sigma_all, state['xi_best'], T, p_u)
    summary = out.groupby('ecozone', as_index=False).mean()
    summary['ecozone_name'] = summary['ecozone'].map(ECOZONE_NAMES).fillna(summary['ecozone'].astype(str))
    cols = ['ecozone', 'ecozone_name', 'rl_2yr', 'rl_5yr', 'rl_10yr', 'rl_20yr']
    summary = summary[cols].sort_values('rl_20yr', ascending=False)
    save_dataframe(summary, 'return_levels_by_ecozone.csv')
    plot_df = summary.head(12).iloc[::-1]
    plt.figure(figsize=(9, 6))
    plt.barh(plot_df['ecozone_name'], plot_df['rl_20yr'])
    plt.xlabel('20-year return level (m/day)')
    plt.ylabel('Ecozone')
    plt.title('Top ecozones by predicted 20-year return level')
    save_open_figure('return_level_20yr_top_ecozones.png')
    return summary


## 16. Choropleth maps

These helpers create ecozone and biome choropleths from the return-level outputs. They require an ecozone polygon file whose zone identifier can be matched to the notebook's `ecozone` values.

The biome choropleth now uses a full national draft crosswalk covering all notebook ecozone codes, while preserving code `999` only as a validation-era convenience label.

In [37]:
import geopandas as gpd

def load_ecozone_boundaries(path: str | None = ECOZONE_SHAPEFILE_PATH, id_column: str = ECOZONE_ID_COLUMN) -> gpd.GeoDataFrame | None:
    if not path:
        print('Skipping choropleths: set ECOZONE_SHAPEFILE_PATH to an ecozone polygon file path.')
        return None
    gdf = gpd.read_file(path)
    if id_column not in gdf.columns:
        raise ValueError(f'ECOZONE_ID_COLUMN={id_column!r} not found in boundary file columns: {list(gdf.columns)}')
    gdf = gdf.copy()
    gdf[id_column] = pd.to_numeric(gdf[id_column], errors='coerce')
    gdf = gdf.dropna(subset=[id_column]).copy()
    gdf[id_column] = gdf[id_column].astype(int)
    return gdf

def plot_ecozone_choropleth(return_level_df: pd.DataFrame, eco_gdf: gpd.GeoDataFrame, value_col: str = 'rl_20yr') -> Path:
    merged = eco_gdf.merge(return_level_df, left_on=ECOZONE_ID_COLUMN, right_on='ecozone', how='left')
    fig, ax = plt.subplots(figsize=(10, 8))
    merged.plot(column=value_col, cmap='YlOrRd', linewidth=0.4, edgecolor='0.4', legend=True, ax=ax, missing_kwds={'color': 'lightgrey', 'label': 'No data'})
    ax.set_title(f'Ecozone choropleth: {value_col}')
    ax.set_axis_off()
    fig.tight_layout()
    path = ensure_output_dir() / f'ecozone_choropleth_{value_col}.png'
    fig.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    merged.drop(columns='geometry').to_csv(ensure_output_dir() / f'ecozone_choropleth_{value_col}.csv', index=False)
    return path

def build_biome_boundaries(eco_gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    eco = eco_gdf.copy()
    if BIOME_NAME_COLUMN and BIOME_NAME_COLUMN in eco.columns:
        eco['biome'] = eco[BIOME_NAME_COLUMN].astype(str)
    else:
        eco['biome'] = eco[ECOZONE_ID_COLUMN].map(BIOME_MAP).fillna('Other')
    biome_gdf = eco[['biome', 'geometry']].dissolve(by='biome', as_index=False)
    return biome_gdf

def plot_biome_choropleth(return_level_df: pd.DataFrame, eco_gdf: gpd.GeoDataFrame, value_col: str = 'rl_20yr') -> Path:
    biome_values = return_level_df.copy()
    biome_values['biome'] = biome_values['ecozone'].map(BIOME_MAP).fillna('Other')
    biome_values = biome_values.groupby('biome', as_index=False)[['rl_2yr', 'rl_5yr', 'rl_10yr', 'rl_20yr']].mean()
    biome_gdf = build_biome_boundaries(eco_gdf)
    merged = biome_gdf.merge(biome_values, on='biome', how='left')
    fig, ax = plt.subplots(figsize=(10, 8))
    merged.plot(column=value_col, cmap='YlGnBu', linewidth=0.5, edgecolor='0.35', legend=True, ax=ax, missing_kwds={'color': 'lightgrey', 'label': 'No data'})
    ax.set_title(f'Biome choropleth: {value_col}')
    ax.set_axis_off()
    fig.tight_layout()
    path = ensure_output_dir() / f'biome_choropleth_{value_col}.png'
    fig.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    merged.drop(columns='geometry').to_csv(ensure_output_dir() / f'biome_choropleth_{value_col}.csv', index=False)
    return path

def run_choropleth_maps(state: dict) -> dict:
    if 'return_level_df' not in state:
        state['return_level_df'] = run_return_level_analysis(state)
    eco_gdf = load_ecozone_boundaries()
    if eco_gdf is None:
        return {}
    outputs = {}
    outputs['ecozone_map'] = plot_ecozone_choropleth(state['return_level_df'], eco_gdf, value_col='rl_20yr')
    outputs['biome_map'] = plot_biome_choropleth(state['return_level_df'], eco_gdf, value_col='rl_20yr')
    return outputs


## 17. Run-all execution

In [19]:
def compute_main_results() -> Dict[str, object]:
    ensure_output_dir()
    df_raw = load_from_bigquery()
    df = preprocess_fire_days(df_raw)
    summarize_dataset(df)

    train_mask, val_mask = make_train_val_masks(df)
    threshold_result = fit_threshold_model(df, ALL_FEATURES, train_mask, val_mask)
    ebm, f_interp = fit_ebm_layer(df, train_mask)
    plot_ebm_effects_to_file(ebm)

    tail_data = make_tail_data(df, threshold_result, f_interp, train_mask, val_mask, ALL_FEATURES)

    # Guardrail: Check if we have enough tail data to fit the model
    min_samples = 50
    if len(tail_data.y_train) < min_samples or len(tail_data.y_val) < min_samples:
        print(f'\nWARNING: Insufficient tail samples (Train: {len(tail_data.y_train)}, Val: {len(tail_data.y_val)}). Skipping GPD modeling.')
        can_fit = False
    else:
        can_fit = True

    if can_fit:
        xi_profile = profile_xi(tail_data.X_train, tail_data.y_train, tail_data.X_val, tail_data.y_val)
        xi_best = float(xi_profile.iloc[0]['xi'])
        save_dataframe(xi_profile, 'xi_profile.csv')

        tail_model = fit_tail_model(tail_data, xi_best)
        X_all_aug = build_augmented_matrix(df, f_interp, ALL_FEATURES)
        sigma_all = predict_sigma(tail_model, X_all_aug)

        y = df[RESPONSE_COL].to_numpy(dtype=float)
        credible_mask = threshold_result.credible_mask
        train_exc_mask = credible_mask & train_mask
        val_exc_mask = credible_mask & val_mask
        y_train_exc = np.minimum(y[train_exc_mask] - threshold_result.threshold[train_exc_mask], threshold_result.cap_value)
        y_val_exc = np.minimum(y[val_exc_mask] - threshold_result.threshold[val_exc_mask], threshold_result.cap_value)
        sigma_val = sigma_all[val_exc_mask]

        stationary_sigma, stationary_xi = stationary_gpd_baseline(y_train_exc)
        const_sigma = fit_constant_sigma(y_train_exc, xi_best)
        full_eval = evaluate_gpd_model(y_val_exc, sigma_val, xi_best, label=f'GBM-GPD fixed xi={xi_best:.2f}')
        b1_eval = evaluate_gpd_model(y_val_exc, np.full_like(y_val_exc, stationary_sigma), stationary_xi, label='Stationary GPD')
        b3_eval = evaluate_gpd_model(y_val_exc, np.full_like(y_val_exc, const_sigma), xi_best, label='Const-sigma GPD')
        evaluation_df = pd.DataFrame([b1_eval, b3_eval, full_eval])
        save_dataframe(evaluation_df, 'gpd_baseline_comparison.csv')
    else:
        xi_profile, xi_best, tail_model, sigma_all, evaluation_df = pd.DataFrame(), np.nan, None, np.full(len(df), np.nan), pd.DataFrame()

    quantile_models = fit_quantile_baselines(df, ALL_FEATURES, train_mask, val_mask, levels=(0.95, 0.99))
    y_val_full = df[RESPONSE_COL].to_numpy(dtype=float)[val_mask]
    q_rows = []
    for tau, model in quantile_models.items():
        q_rows.append(evaluate_quantile_predictions(y_val_full, model.predict(df.loc[val_mask, ALL_FEATURES]), tau))
    quantile_df = pd.DataFrame(q_rows)
    save_dataframe(quantile_df, 'quantile_baseline_comparison.csv')

    return {
        'df': df, 'train_mask': train_mask, 'val_mask': val_mask, 'threshold_result': threshold_result,
        'ebm': ebm, 'f_interp': f_interp, 'tail_data': tail_data, 'xi_best': xi_best, 'tail_model': tail_model,
        'sigma_all': sigma_all, 'y': df[RESPONSE_COL].to_numpy(dtype=float), 'evaluation_df': evaluation_df,
        'quantile_df': quantile_df, 'can_fit_gpd': can_fit
    }

# Re-run the main analysis flow
analysis_state = compute_main_results()

Clean dataset: 45,369 fire-days
Median spread: 356 m/day
95th percentile spread: 3342 m/day
Max spread: 26925 m/day
Interpreted features: 7
Non-interpreted features: 42



In [20]:
def check_tail_counts(df, threshold_result, train_mask, val_mask):
    u = threshold_result.threshold
    credible = threshold_result.credible_mask

    train_exc = df[credible & train_mask]
    val_exc = df[credible & val_mask]

    print(f'Total rows in df: {len(df)}')
    print(f'Rows meeting threshold (credible_mask): {credible.sum()}')
    print(f'Tail samples for Training: {len(train_exc)}')
    print(f'Tail samples for Validation: {len(val_exc)}')

    if len(train_exc) == 0 or len(val_exc) == 0:
        print('\nALERT: One of your tail datasets is empty. Lower EXCEEDANCE_FLOOR or TAU to include more samples.')

# Run this after threshold_result is created in your flow
# check_tail_counts(df, threshold_result, train_mask, val_mask)

## 18. Notes

This notebook replaces the old manual execution fragments with one explicit orchestration path. The original notebook is still useful as a reference, but the refactored version is intended to be the version you use for top-to-bottom reruns and figure regeneration.